#  IMPORTS & ENVIRONMENT SETUP

In [ ]:
# ============================================================
# CELL 01: IMPORTS & LOGGER
# ============================================================
import os, json, time, random, shutil, logging, xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image as PILImage
from ultralytics import YOLO
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
# ✅ SỬA: torch.cuda.amp.GradScaler deprecated → dùng torch.amp
from torch.amp import GradScaler, autocast

import torchvision
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    ssd300_vgg16,
    retinanet_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
    SSD300_VGG16_Weights,
    RetinaNet_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.models.detection.ssd import SSDClassificationHead
import torchvision.ops as ops
import os, time, yaml, random
import albumentations as A
from albumentations.pytorch import ToTensorV2

from torchmetrics.detection.mean_ap import MeanAveragePrecision
from sklearn.metrics import confusion_matrix

# ─── Logger ───────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("RDD2022")

# ─── GPU Setup (NVIDIA GeForce RTX 5060 8GB) ──────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram     = torch.cuda.get_device_properties(0).total_memory / 1e9
    logger.info(f"✅ GPU: {gpu_name} | VRAM: {vram:.1f} GB")
    torch.backends.cudnn.benchmark     = True
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    logger.warning("⚠️  Không tìm thấy GPU — chạy trên CPU")

logger.info(f"PyTorch: {torch.__version__} | torchvision: {torchvision.__version__} | Device: {device}")


## PATHS & HYPERPARAMETERS

In [ ]:
# ============================================================
# CELL 02: PATHS & HYPERPARAMETERS CONFIGURATION
# ============================================================

# ─── Root paths ───────────────────────────────────────────
BDD_ROOT      = Path(r"D:\Code\Data\RDDD47k_70_20_10")
TRAIN_IMG_DIR = BDD_ROOT / "train" / "images"
TRAIN_LBL_DIR = BDD_ROOT / "train" / "labels"
VALID_IMG_DIR = BDD_ROOT / "valid" / "images"
VALID_LBL_DIR = BDD_ROOT / "valid" / "labels"
TEST_IMG_DIR  = BDD_ROOT / "test"  / "images"
TEST_LBL_DIR  = BDD_ROOT / "test"  / "labels"
DATA_YAML_PATH = BDD_ROOT / "road_damage.yaml"

# ─── Output directories ───────────────────────────────────
OUTPUT_DIR = BDD_ROOT / "outputs"
CKPT_DIR   = OUTPUT_DIR / "checkpoints"
LOG_DIR    = OUTPUT_DIR / "logs"
RESULT_DIR = OUTPUT_DIR / "results"
RUNS_DIR   = OUTPUT_DIR / "runs"
for d in [OUTPUT_DIR, CKPT_DIR, LOG_DIR, RESULT_DIR, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─── Class mapping ────────────────────────────────────────
CLASS_NAMES = ["D00", "D10", "D20", "D40"]
# D00=Longitudinal crack | D10=Transverse crack
# D20=Alligator crack    | D40=Pothole
NUM_CLASSES = len(CLASS_NAMES)

# ─── Tỉ lệ chia dữ liệu ──────────────────────────────────
TRAIN_RATIO = 0.70
VALID_RATIO = 0.20
TEST_RATIO  = 0.10

HPARAMS = {
    'numepochs': 100,      # Tăng từ 50 → 100 epochs max mAP50
    'batchsize': 16,       # Tăng từ 8 → 16 (test VRAM 6GB trước)
    'numworkers': 4,
    'optimizer': 'AdamW',
    'learningrate': 0.01,  # TĂNG LR0 từ 0.001 → 0.01 hội tụ nhanh
    'lrfinal': 1e-5,
    'weightdecay': 0.0005,
    'momentum': 0.937,
    'warmupepochs': 3,
    'scheduler': 'cosine',
    'imgsize': 800,
    'augprob': 0.5,
    'patience': 15,
    'seed': 42,
    'amp': True,
    'gradclip': 10.0,
    'confthres': 0.001,
    'iouthres': 0.6,
    'saveperiod': 20,
    'box': 7.5,            # Giữ loss weights
    'cls': 1.5,
    'dfl': 1.5
}


# ─── Reproducibility ──────────────────────────────────────
random.seed(HPARAMS["seed"])
np.random.seed(HPARAMS["seed"])
torch.manual_seed(HPARAMS["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(HPARAMS["seed"])
    torch.backends.cudnn.benchmark    = True
    torch.backends.cuda.matmul.allow_tf32 = True

# ─── Auto cache mode ──────────────────────────────────────
import psutil
_ram_free = psutil.virtual_memory().available / 1e9
CACHE_MODE = "ram" if _ram_free >= 10.0 else False
logger.info(f"RAM free: {_ram_free:.1f}GB → cache={CACHE_MODE}")

# ─── Lưu hyperparameters ra JSON ──────────────────────────
hparam_path = LOG_DIR / "hyperparameters.json"
with open(hparam_path, "w", encoding="utf-8") as f:
    json.dump(
        {**HPARAMS, "num_classes": NUM_CLASSES,
         "class_names": CLASS_NAMES, "cache": CACHE_MODE},
        f, indent=4, ensure_ascii=False,
    )
logger.info(f"📝 Hyperparameters saved → {hparam_path}")

print("\n=== HYPERPARAMETERS ===")
for k, v in HPARAMS.items():
    print(f"  {k:<25s}: {v}")
print(f"  {'cache':<25s}: {CACHE_MODE}")

##  TẠO DATA.YAML CHO ULTRALYTICS YOLO

In [ ]:
# ============================================================
# CELL 03: TẠO DATA.YAML CHO ULTRALYTICS YOLO
# ============================================================
DATA_YAML_PATH = BDD_ROOT / "road_damage.yaml"

data_yaml = {
    "path"  : str(BDD_ROOT),
    "train" : "train/images",
    "val"   : "valid/images",
    "test"  : "test/images",
    "nc"    : NUM_CLASSES,
    "names" : CLASS_NAMES,
}

with open(DATA_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

logger.info(f"✅ data.yaml saved → {DATA_YAML_PATH}")
print("\n=== DATA.YAML CONTENT ===")
print(open(DATA_YAML_PATH).read())


### KIỂM TRA PHÂN PHỐI CLASS

In [ ]:
# ============================================================
# CELL 04: KIỂM TRA SỐ LƯỢNG NHÃN (YOLO .txt format)
# ============================================================

def count_yolo_labels(lbl_dir: Path) -> tuple:
    """Đếm số bounding box theo class_id trong thư mục YOLO .txt"""
    counts      = defaultdict(int)
    total_files = 0
    empty_files = 0
    for txt_file in lbl_dir.glob("*.txt"):
        total_files += 1
        lines = txt_file.read_text().strip().splitlines()
        if not lines or lines == [""]:
            empty_files += 1
            continue
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 5:
                counts[int(parts[0])] += 1
    return dict(counts), total_files, empty_files


splits = {
    "Train (70%)": TRAIN_LBL_DIR,
    "Valid (20%)": VALID_LBL_DIR,
    "Test  (10%)": TEST_LBL_DIR,
}

all_counts, all_files, all_empty = {}, {}, {}
for split_name, lbl_dir in splits.items():
    c, nf, ne          = count_yolo_labels(lbl_dir)
    all_counts[split_name] = c
    all_files[split_name]  = nf
    all_empty[split_name]  = ne

tr, va, te = "Train (70%)", "Valid (20%)", "Test  (10%)"
totals = {s: sum(c.values()) for s, c in all_counts.items()}

# ── In bảng thống kê ──────────────────────────────────────
print("\n" + "=" * 75)
print(f"  {'Class':<10} {'Train':>8} {'Train%':>7}  {'Valid':>7} {'Valid%':>7}  {'Test':>7} {'Test%':>7}")
print("-" * 75)

for cls_id, cls_name in enumerate(CLASS_NAMES):
    t = all_counts[tr].get(cls_id, 0)
    v = all_counts[va].get(cls_id, 0)
    e = all_counts[te].get(cls_id, 0)
    print(
        f"  {cls_name:<10} {t:>8,} {t/max(totals[tr],1)*100:>6.1f}%"
        f"  {v:>7,} {v/max(totals[va],1)*100:>6.1f}%"
        f"  {e:>7,} {e/max(totals[te],1)*100:>6.1f}%"
    )

print("-" * 75)
for s, tot in totals.items():
    print(f"  {'TOTAL ' + s:<30} {tot:>8,} boxes")
print("-" * 75)
for s in [tr, va, te]:
    print(f"  {'FILES ' + s:<30} {all_files[s]:>8,} files  |  empty: {all_empty[s]:,}")
print("=" * 75)

grand_total = sum(totals.values())
grand_files = sum(all_files.values())
if grand_total > 0:
    print(f"\n📊 Tỉ lệ split (theo bbox):")
    for s, tot in totals.items():
        print(f"   {s:<14}: {tot:>7,}  ({tot/grand_total*100:.1f}%)")
    print(f"\n📁 Tỉ lệ split (theo file ảnh):")
    for s, nf in all_files.items():
        print(f"   {s:<14}: {nf:>7,}  ({nf/grand_files*100:.1f}%)")
else:
    print("⚠️  Không tìm thấy nhãn — kiểm tra lại đường dẫn BDD_ROOT!")


In [ ]:
# ============================================================
# RDD2022 - Dataset Explorer
# Cấu trúc: D:\Code\Data\RDDD47k_70_20_10
# Labels: YOLO format (.txt), 4 classes: D00 D10 D20 D40
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")

# ── Paths ────────────────────────────────────────────────────
BDD_ROOT      = Path(r"D:\Code\Data\RDDD47k_70_20_10")
TRAIN_LBL_DIR = BDD_ROOT / "train" / "labels"
VALID_LBL_DIR = BDD_ROOT / "valid" / "labels"
TEST_LBL_DIR  = BDD_ROOT / "test"  / "labels"
TRAIN_IMG_DIR = BDD_ROOT / "train" / "images"
VALID_IMG_DIR = BDD_ROOT / "valid" / "images"
TEST_IMG_DIR  = BDD_ROOT / "test"  / "images"

CLASS_NAMES = {0: "D00", 1: "D10", 2: "D20", 3: "D40"}
SPLITS      = {"train": TRAIN_LBL_DIR,
               "valid": VALID_LBL_DIR,
               "test" : TEST_LBL_DIR}
IMG_SPLITS  = {"train": TRAIN_IMG_DIR,
               "valid": VALID_IMG_DIR,
               "test" : TEST_IMG_DIR}

# ── Hàm nhận diện quốc gia từ tên file ──────────────────────
COUNTRY_MAP = {
    "japan"  : "Japan",    "jpn": "Japan",
    "india"  : "India",    "ind": "India",
    "czech"  : "Czech",    "cze": "Czech",
    "norway" : "Norway",   "nor": "Norway",
    "united" : "USA",      "usa": "USA",      "us_": "USA",
    "china"  : "China",    "chn": "China",    "drone": "China",
    "motorbike": "China",
}

def detect_country(name: str) -> str:
    n = name.lower()
    for key, country in COUNTRY_MAP.items():
        if key in n:
            return country
    return "Unknown"

# ── Parse toàn bộ label ──────────────────────────────────────
records = []
for split, lbl_dir in SPLITS.items():
    if not lbl_dir.exists():
        print(f"[WARN] Không tìm thấy: {lbl_dir}")
        continue
    for txt in lbl_dir.glob("*.txt"):
        country = detect_country(txt.stem)
        with open(txt) as f:
            lines = [l.strip() for l in f if l.strip()]
        if lines:
            for line in lines:
                cls_id = int(line.split()[0])
                records.append({"split": split, "country": country,
                                 "class_id": cls_id,
                                 "class": CLASS_NAMES.get(cls_id, f"cls{cls_id}"),
                                 "file": txt.stem})
        else:
            # ảnh không có nhãn (negative)
            records.append({"split": split, "country": country,
                             "class_id": -1, "class": "No damage",
                             "file": txt.stem})

df = pd.DataFrame(records)
print(f"Tổng số bounding box (có nhãn): {len(df[df.class_id >= 0]):,}")
print(f"Tổng số ảnh unique: {df.file.nunique():,}")
print(df.groupby("split")["file"].nunique().rename("images").to_frame())

# ── Đếm ảnh theo split (kể cả ảnh không có label .txt) ───────
img_counts = {s: len(list(d.glob("*.jpg")) + list(d.glob("*.png")))
              for s, d in IMG_SPLITS.items() if d.exists()}
print("\nSố ảnh theo thư mục images:")
for s, c in img_counts.items():
    print(f"  {s:>6}: {c:,}")

# ── Màu sắc ─────────────────────────────────────────────────
C_CLASS   = ["#4C9BE8","#F4A952","#6DBF6D","#E86C6C"]
C_COUNTRY = ["#3A86FF","#FF6B6B","#FFBE0B","#06D6A0","#8338EC","#FB5607"]
C_SPLIT   = {"train":"#3A86FF","valid":"#FFBE0B","test":"#FF6B6B"}

fig = plt.figure(figsize=(20, 22))
fig.suptitle("RDD2022 — Phân tích bộ dữ liệu", fontsize=18, fontweight="bold", y=0.98)

# ── [1] Phân bố class toàn bộ dataset ───────────────────────
ax1 = fig.add_subplot(3, 3, 1)
cls_all = df[df.class_id >= 0]["class"].value_counts().reindex(CLASS_NAMES.values(), fill_value=0)
bars = ax1.bar(cls_all.index, cls_all.values, color=C_CLASS, edgecolor="white", width=0.6)
ax1.bar_label(bars, fmt="{:,.0f}", padding=3, fontsize=9)
ax1.set_title("Phân bố class — Toàn bộ dataset", fontweight="bold")
ax1.set_ylabel("Số bounding box")
ax1.set_xlabel("Class")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# ── [2] Pie chart class toàn bộ ─────────────────────────────
ax2 = fig.add_subplot(3, 3, 2)
ax2.pie(cls_all.values, labels=cls_all.index, autopct="%1.1f%%",
        colors=C_CLASS, startangle=140, pctdistance=0.82,
        wedgeprops=dict(edgecolor="white", linewidth=1.5))
ax2.set_title("Tỉ lệ class — Toàn bộ dataset", fontweight="bold")

# ── [3] Số ảnh theo split ────────────────────────────────────
ax3 = fig.add_subplot(3, 3, 3)
split_img = pd.Series(img_counts)
colors3 = [C_SPLIT.get(s, "#aaa") for s in split_img.index]
bars3 = ax3.bar(split_img.index, split_img.values, color=colors3, edgecolor="white", width=0.5)
ax3.bar_label(bars3, fmt="{:,.0f}", padding=3, fontsize=10)
ax3.set_title("Số ảnh theo Split", fontweight="bold")
ax3.set_ylabel("Số ảnh")
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# ── [4] Phân bố class theo split (grouped bar) ──────────────
ax4 = fig.add_subplot(3, 3, 4)
cls_split = df[df.class_id >= 0].groupby(["split","class"]).size().unstack(fill_value=0)
cls_split = cls_split.reindex(columns=CLASS_NAMES.values(), fill_value=0)
x  = np.arange(len(cls_split))
bw = 0.2
for i, cls in enumerate(CLASS_NAMES.values()):
    ax4.bar(x + i*bw, cls_split[cls], width=bw, label=cls,
            color=C_CLASS[i], edgecolor="white")
ax4.set_xticks(x + bw*1.5)
ax4.set_xticklabels(cls_split.index)
ax4.legend(title="Class", fontsize=8)
ax4.set_title("Phân bố class theo Split", fontweight="bold")
ax4.set_ylabel("Số bounding box")
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# ── [5] Stacked bar class theo split ────────────────────────
ax5 = fig.add_subplot(3, 3, 5)
bottom = np.zeros(len(cls_split))
for i, cls in enumerate(CLASS_NAMES.values()):
    ax5.bar(cls_split.index, cls_split[cls], bottom=bottom,
            label=cls, color=C_CLASS[i], edgecolor="white")
    bottom += cls_split[cls].values
ax5.legend(title="Class", fontsize=8)
ax5.set_title("Stacked — Class theo Split", fontweight="bold")
ax5.set_ylabel("Số bounding box")
ax5.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# ── [6] Phân bố ảnh theo quốc gia (toàn bộ) ─────────────────
ax6 = fig.add_subplot(3, 3, 6)
country_img = df.drop_duplicates("file").groupby("country").size().sort_values(ascending=False)
bars6 = ax6.bar(country_img.index, country_img.values,
                color=C_COUNTRY[:len(country_img)], edgecolor="white")
ax6.bar_label(bars6, fmt="{:,.0f}", padding=3, fontsize=9)
ax6.set_title("Số ảnh theo Quốc gia — Toàn bộ", fontweight="bold")
ax6.set_ylabel("Số ảnh")
ax6.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# ── [7] Phân bố ảnh theo quốc gia × split ───────────────────
ax7 = fig.add_subplot(3, 3, 7)
country_split = df.drop_duplicates("file").groupby(["split","country"]).size().unstack(fill_value=0)
countries = country_split.columns.tolist()
x7 = np.arange(len(country_split))
bw7 = 0.8 / len(countries)
for i, c in enumerate(countries):
    ax7.bar(x7 + i*bw7, country_split[c], width=bw7, label=c,
            color=C_COUNTRY[i % len(C_COUNTRY)], edgecolor="white")
ax7.set_xticks(x7 + bw7*(len(countries)-1)/2)
ax7.set_xticklabels(country_split.index)
ax7.legend(title="Country", fontsize=7, ncol=2)
ax7.set_title("Ảnh theo Quốc gia × Split", fontweight="bold")
ax7.set_ylabel("Số ảnh")
ax7.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# ── [8] Class distribution theo quốc gia ────────────────────
ax8 = fig.add_subplot(3, 3, 8)
cls_country = df[df.class_id >= 0].groupby(["country","class"]).size().unstack(fill_value=0)
cls_country = cls_country.reindex(columns=CLASS_NAMES.values(), fill_value=0)
bottom8 = np.zeros(len(cls_country))
for i, cls in enumerate(CLASS_NAMES.values()):
    ax8.bar(cls_country.index, cls_country[cls], bottom=bottom8,
            label=cls, color=C_CLASS[i], edgecolor="white")
    bottom8 += cls_country[cls].values
ax8.legend(title="Class", fontsize=8)
ax8.set_title("Class theo Quốc gia (stacked)", fontweight="bold")
ax8.set_ylabel("Số bounding box")
plt.setp(ax8.get_xticklabels(), rotation=20, ha="right")
ax8.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# ── [9] Heatmap: class × country ────────────────────────────
ax9 = fig.add_subplot(3, 3, 9)
heatmap_data = cls_country.T
im = ax9.imshow(heatmap_data.values, aspect="auto", cmap="YlOrRd")
ax9.set_xticks(range(len(heatmap_data.columns)))
ax9.set_xticklabels(heatmap_data.columns, rotation=25, ha="right", fontsize=8)
ax9.set_yticks(range(len(heatmap_data.index)))
ax9.set_yticklabels(heatmap_data.index)
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        v = heatmap_data.values[i, j]
        ax9.text(j, i, f"{v:,}", ha="center", va="center",
                 fontsize=7, color="black" if v < heatmap_data.values.max()*0.6 else "white")
plt.colorbar(im, ax=ax9, shrink=0.8)
ax9.set_title("Heatmap: Class × Quốc gia", fontweight="bold")

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig("rdd2022_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Đã lưu biểu đồ: rdd2022_analysis.png")

# ── In bảng thống kê tổng hợp ────────────────────────────────
print("\n📊 Phân bố class toàn bộ dataset:")
print(cls_all.to_string())
print("\n📊 Phân bố class theo split:")
print(cls_split.to_string())
print("\n📊 Số ảnh theo quốc gia × split:")
print(country_split.to_string())

### DATA AUGMENTATION & PREPROCESSING

In [ ]:
# ============================================================
# CELL 05: VISUALIZE PHÂN BỐ NHÃN
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("RDD2022 — Phân bố nhãn theo split", fontsize=13, fontweight="bold")

x = np.arange(NUM_CLASSES)
w = 0.25
split_colors = {tr: "#e74c3c", va: "#3498db", te: "#2ecc71"}

for i, (split_name, counts) in enumerate(all_counts.items()):
    vals = [counts.get(c, 0) for c in range(NUM_CLASSES)]
    axes[0].bar(x + i * w, vals, w,
                label=split_name, color=split_colors[split_name], alpha=0.85)

axes[0].set_xticks(x + w)
axes[0].set_xticklabels(CLASS_NAMES, fontsize=11)
axes[0].set_title("Số bounding box theo class & split")
axes[0].set_ylabel("Số lượng bounding boxes")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Pie chart — Train set
pie_vals = [all_counts[tr].get(c, 0) for c in range(NUM_CLASSES)]
pie_lbls = [f"{CLASS_NAMES[c]}\n{v:,}" for c, v in enumerate(pie_vals)]
axes[1].pie(
    pie_vals, labels=pie_lbls, autopct="%1.1f%%",
    colors=["#e74c3c", "#3498db", "#2ecc71", "#f39c12"],
    startangle=140, textprops={"fontsize": 10},
)
axes[1].set_title("Tỉ lệ class trong Train set")

plt.tight_layout()
save_path = RESULT_DIR / "label_distribution.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()
logger.info(f"📊 Saved → {save_path}")


### DATASET & DATALOADER

In [ ]:
# ============================================================
# CELL 06: VISUALIZE ANNOTATIONS (YOLO .txt → bounding boxes)
# ============================================================
COLORS_VIZ = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12"]

def visualize_yolo_labels(img_dir: Path, lbl_dir: Path,
                           n: int = 6, split_name: str = "TRAIN"):
    img_paths = sorted(list(img_dir.glob("*.jpg")) +
                       list(img_dir.glob("*.png")))
    img_paths = random.sample(img_paths, min(n, len(img_paths)))

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Visualize Annotations — {split_name}",
                 fontsize=14, fontweight="bold")

    for ax, img_path in zip(axes.flatten(), img_paths):
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        ax.imshow(img)

        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id          = int(parts[0])
                cx, cy, bw, bh  = map(float, parts[1:5])
                x1 = (cx - bw / 2) * W
                y1 = (cy - bh / 2) * H
                color = COLORS_VIZ[cls_id % len(COLORS_VIZ)]
                rect  = patches.Rectangle(
                    (x1, y1), bw * W, bh * H,
                    linewidth=2, edgecolor=color, facecolor="none",
                )
                ax.add_patch(rect)
                ax.text(x1, y1 - 4, CLASS_NAMES[cls_id],
                        color=color, fontsize=8, fontweight="bold",
                        bbox=dict(facecolor="black", alpha=0.4, pad=1))
        ax.set_title(img_path.name, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    out = RESULT_DIR / f"viz_labels_{split_name.lower()}.png"
    plt.savefig(out, dpi=130, bbox_inches="tight")
    plt.show()
    logger.info(f"📊 Saved → {out}")


visualize_yolo_labels(TRAIN_IMG_DIR, TRAIN_LBL_DIR, split_name="TRAIN")
visualize_yolo_labels(VALID_IMG_DIR, VALID_LBL_DIR, split_name="VALID")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# DATA AUGMENTATION CONFIG MỚI (Fix background/D00 confusion)
# Lý do chọn từng tham số:
#
#  mosaic=1.0      : Quan trọng nhất cho crack nhỏ — ghép 4 ảnh
#                     giúp model thấy crack ở nhiều context
#                     (tắt 10 epoch cuối qua close_mosaic=10)
#  mixup=0.0       : ❌ TẮT — tránh blend nhầm D00/background
#  fliplr=0.5      : Crack xuất hiện cả 2 hướng → nên bật
#  flipud=0.0      : ❌ Tắt — mặt đường không lật dọc (không thực tế)
#  hsv_s=0.7       : Color jitter mạnh → handle các loại camera khác nhau
#                     (RDD2022 gồm ảnh từ Nhật, Ấn Độ, Na Uy, Czech)
#  scale=0.5       : Random zoom in/out → crack ở nhiều khoảng cách
#  copy_paste=0.1  : Nhẹ — copy bbox sang ảnh khác, tốt cho minority class (D40)
#  erasing=0.0     : ❌ TẮT — tránh xóa boundary crack/background
#  degrees=5.0     : Xoay nhẹ — camera hơi nghiêng khi lái
#  shear=2.0       : Distortion nhẹ — perspective thay đổi
#  perspective=0.0005: Rất nhẹ, đủ để simulate góc camera khác nhau
# ──────────────────────────────────────────────────────────────
AUG_CONFIG = {
    # Spatial
    "mosaic"       : 1.0,    # ↑ 0.0→1.0: BẬT — quan trọng nhất cho crack nhỏ
    "mixup"        : 0.0,    # ↓ 0.1→0.0: TẮT — fix background/D00 confusion
    "close_mosaic" : 10,     # Tắt mosaic 10 epoch cuối → stabilize

    # Flip
    "fliplr"       : 0.5,    # Giữ nguyên: crack đối xứng
    "flipud"       : 0.0,    # ✅ Tắt — mặt đường không lật dọc

    # Color
    "hsv_h"        : 0.015,
    "hsv_s"        : 0.7,    # Mạnh: handle multi-camera (RDD2022)
    "hsv_v"        : 0.4,

    # Geometric
    "scale"        : 0.5,    # Zoom in/out: crack nhiều khoảng cách
    "translate"    : 0.1,
    "degrees"      : 5.0,    # Xoay nhẹ: camera nghiêng
    "shear"        : 2.0,    # Distortion: perspective
    "perspective"  : 0.0005, # Rất nhẹ: góc camera khác nhau

    # Copy-Paste & Erasing
    "copy_paste"   : 0.1,    # Nhẹ: tốt cho D40 minority
    "erasing"      : 0.0     # ↓ 0.4→0.0: TẮT — tránh xóa boundary D00
}

# ═══════════════════════════════════════════════════════════════
# Print config đẹp + status
print("═" * 70)
print("✅ DATA AUGMENTATION CONFIG MỚI (Fix 2374 BG→D00, 1313 D00→BG)")
print("═" * 70)

aug_desc = {
    "mosaic":       "1. Mosaic ghép 4 ảnh → crack nhiều context",
    "mixup":        "2. Mixup blend 2 ảnh → ❌ TẮT (fix confusion)",
    "close_mosaic": "3. Close Mosaic 10 epochs cuối → stabilize",
    "fliplr":       "4. Horizontal Flip → crack 2 hướng",
    "flipud":       "5. Vertical Flip → ❌ Tắt (không thực tế)",
    "hsv_h":        "6. HSV Hue jitter",
    "hsv_s":        "7. HSV Saturation → mạnh multi-camera",
    "hsv_v":        "8. HSV Value jitter",
    "scale":        "9. Random Scale zoom/inout",
    "copy_paste":   "10. Copy-Paste minority class (D40)",
    "erasing":      "11. Random Erase Cutout → ❌ TẮT (fix boundary)"
}

for k, desc in aug_desc.items():
    status = "✅ BẬT" if AUG_CONFIG[k] > 0 else "❌ TẮT"
    print(f"{status:>8} | {desc:<45} | {k}: {AUG_CONFIG[k]}")

print("═" * 70)
logger.info("AUGCONFIG ready - Optimized for road crack detection")

In [ ]:
# ============================================================
# CELL 08: UTILITY — LOG HYPERPARAMETERS & TRAINING TIME
# ============================================================

def log_experiment(model_name: str, hp: dict, aug: dict,
                   train_time_sec: float, results_dir: Path) -> dict:
    """Ghi lại toàn bộ siêu tham số, thời gian và kết quả ra JSON."""
    log = {
        "model"           : model_name,
        "timestamp"       : datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "train_time_sec"  : round(train_time_sec, 2),
        "train_time_min"  : round(train_time_sec / 60, 2),
        "hyperparameters" : hp,
        "augmentation"    : aug,
        "results_dir"     : str(results_dir),
    }
    fname = LOG_DIR / f"log_{model_name}_{datetime.now().strftime('%H%M%S')}.json"
    with open(fname, "w", encoding="utf-8") as f:
        json.dump(log, f, indent=4, ensure_ascii=False)
    logger.info(f"📋 Experiment log saved → {fname}")
    return log


def print_training_summary(model_name: str, elapsed_sec: float):
    print(f"\n{'='*55}")
    print(f"  ✅ {model_name} — Huấn luyện hoàn tất")
    print(f"  ⏱  Thời gian: {elapsed_sec:.1f}s  ({elapsed_sec/60:.2f} phút)")
    print(f"{'='*55}")


logger.info("✅ Utility functions ready")


### ĐÁNH TRỌNG SỐ — 3 CẤP ĐỘ

In [ ]:
# ============================================================
# CELL 08b: ĐÁNH TRỌNG SỐ — 2 CẤP ĐỘ (ĐÃ SỬA)
#   1. Loss weights  (box / cls / dfl)
#   2. Class weights tự động từ phân bố nhãn
#   ❌ Đã xóa: Weighted DataLoader (monkey-patch → PicklingError)
# ============================================================

import numpy as np
from collections import defaultdict
from pathlib import Path

# ─────────────────────────────────────────────────────────────
# PHẦN 1: LOSS WEIGHTS (box / cls / dfl)
# ─────────────────────────────────────────────────────────────
LOSS_WEIGHTS = {
    "box": 7.5,   # Bounding box regression — giữ default
    "cls": 1.5,   # Classification — tăng do class imbalance RDD2022
    "dfl": 1.5,   # Distribution Focal Loss — giữ default
}
HPARAMS.update(LOSS_WEIGHTS)

print("=== LOSS WEIGHTS ===")
print(f"  box (bbox regression)  : {LOSS_WEIGHTS['box']}  (default=7.5)")
print(f"  cls (classification)   : {LOSS_WEIGHTS['cls']}  (default=0.5 → tăng do imbalance)")
print(f"  dfl (focal loss dist)  : {LOSS_WEIGHTS['dfl']}  (default=1.5)")

# ─────────────────────────────────────────────────────────────
# PHẦN 2: TÍNH CLASS WEIGHTS TỰ ĐỘNG (Inverse Frequency)
# ─────────────────────────────────────────────────────────────
def compute_class_weights(lbl_dir: Path, num_classes: int):
    """
    Tính class weight = total / (num_classes × count_per_class)
    → Class ít mẫu → weight cao → model chú ý hơn
    """
    counts = np.zeros(num_classes, dtype=np.float32)
    for txt_file in lbl_dir.glob("*.txt"):
        for line in txt_file.read_text().strip().splitlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(parts[0])
                if cls_id < num_classes:
                    counts[cls_id] += 1

    counts  = np.where(counts == 0, 1, counts)   # tránh chia 0
    total   = counts.sum()
    weights = total / (num_classes * counts)
    weights = weights / weights.mean()            # normalize: mean = 1.0
    return weights, counts


class_weights, class_counts = compute_class_weights(TRAIN_LBL_DIR, NUM_CLASSES)

print("\n=== CLASS WEIGHTS (Inverse Frequency) ===")
print(f"  {'Class':<8} {'Count':>8} {'Weight':>10}  {'Mức độ'}")
print(f"  {'-'*50}")
for name, cnt, w in zip(CLASS_NAMES, class_counts, class_weights):
    level = "⬆ Ưu tiên cao" if w > 1.2 else ("⬇ Giảm trọng" if w < 0.8 else "✓ Cân bằng")
    print(f"  {name:<8} {int(cnt):>8,} {w:>10.4f}  {level}")

print(f"\n  Tổng bbox train  : {int(class_counts.sum()):,}")
print(f"  Imbalance ratio  : {class_counts.max() / class_counts.min():.2f}x")

# ── Lưu class weights ra JSON ─────────────────────────────
cw_path = LOG_DIR / "class_weights.json"
with open(cw_path, "w") as f:
    json.dump({
        "class_names"   : CLASS_NAMES,
        "class_counts"  : class_counts.tolist(),
        "class_weights" : class_weights.tolist(),
        "method"        : "inverse_frequency_normalized",
    }, f, indent=4)
logger.info(f"📋 Class weights saved → {cw_path}")

# ─────────────────────────────────────────────────────────────
# PHẦN 3 (ĐÃ XÓA): Weighted DataLoader / Monkey-patch
# ─────────────────────────────────────────────────────────────
# ❌ KHÔNG làm:
#    _dataset.YOLODataset = YOLOWeightedDataset
#
# Lý do xóa:
#   1. Gây PicklingError trên Windows + Jupyter (workers > 0)
#      → build.py đã cache YOLODataset trước khi patch
#      → spawn worker thấy 2 bản class khác nhau → lỗi
#   2. Class định nghĩa trong Jupyter cell có __module__='__main__'
#      → worker process mới không tìm thấy → TypeError
#   3. Patch không có hiệu lực thực tế vì build_dataloader()
#      dùng reference cũ đã import sẵn
#
# ✅ THAY THẾ: dùng Loss Weights (cls=1.5) + AUG_CONFIG tăng
#    cường augmentation cho minority class — đủ hiệu quả
# ─────────────────────────────────────────────────────────────

# ─────────────────────────────────────────────────────────────
# VISUALIZE: So sánh class counts & weights
# ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Class Weighting Analysis — RDD2022", fontsize=13, fontweight="bold")

colors_bar = [
    "#e74c3c" if w > 1.2 else ("#3498db" if w < 0.8 else "#2ecc71")
    for w in class_weights
]

# Biểu đồ 1: Số lượng bbox per class
axes[0].bar(CLASS_NAMES, class_counts, color=colors_bar, alpha=0.85, edgecolor="black")
axes[0].set_title("Số lượng BBox per class (Train)")
axes[0].set_ylabel("Số bbox")
for i, v in enumerate(class_counts):
    axes[0].text(i, v + max(class_counts) * 0.01, f"{int(v):,}",
                 ha="center", fontsize=10, fontweight="bold")
axes[0].grid(axis="y", alpha=0.3)

# Biểu đồ 2: Class weights
axes[1].bar(CLASS_NAMES, class_weights, color=colors_bar, alpha=0.85, edgecolor="black")
axes[1].axhline(y=1.0, color="black", linestyle="--", linewidth=1.5, label="baseline=1.0")
axes[1].set_title("Class Weights (Inverse Frequency)")
axes[1].set_ylabel("Weight")
for i, w in enumerate(class_weights):
    axes[1].text(i, w + 0.02, f"{w:.3f}", ha="center", fontsize=10, fontweight="bold")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
cw_fig_path = RESULT_DIR / "class_weights.png"
plt.savefig(cw_fig_path, dpi=150, bbox_inches="tight")
plt.show()
logger.info(f"📊 Class weights chart → {cw_fig_path}")

# ─────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────
print("\n✅ Đánh trọng số hoàn tất — sẵn sàng training!")
print(f"   Loss weights  : box={LOSS_WEIGHTS['box']}, "
      f"cls={LOSS_WEIGHTS['cls']}, dfl={LOSS_WEIGHTS['dfl']}")
print(f"   Class weights : {dict(zip(CLASS_NAMES, [f'{w:.3f}' for w in class_weights]))}")
print(f"\n   ℹ️  Weighted DataLoader đã bị vô hiệu hoá")
print(f"      (monkey-patch không tương thích Windows + Jupyter + workers > 0)")

# train yolov8m

In [ ]:
# ============================================================
# CELL 09: TRAIN YOLOv8m
# ============================================================
from ultralytics import YOLO

# ─── Config ───────────────────────────────────────────────
MODEL_NAME  = "yolov8m"
RUN_NAME    = f"{MODEL_NAME}_rdd2022"
WEIGHTS     = "yolov8m.pt"          # pretrained COCO → fine-tune

# ─── Khởi tạo model ───────────────────────────────────────
model_v8m = YOLO(WEIGHTS)
logger.info(f"✅ Loaded: {WEIGHTS}")

# ─── Train ────────────────────────────────────────────────
t0 = time.time()

results_v8m = model_v8m.train(
    # Dataset
    data        = str(DATA_YAML_PATH),
    # Training core
    epochs      = HPARAMS["num_epochs"],
    batch       = HPARAMS["batch_size"],
    imgsz       = HPARAMS["img_size"],
    # Optimizer
    optimizer   = HPARAMS["optimizer"],
    lr0         = HPARAMS["learning_rate"],
    lrf         = HPARAMS["lr_final"],
    weight_decay= HPARAMS["weight_decay"],
    momentum    = HPARAMS["momentum"],
    # Scheduler
    warmup_epochs = HPARAMS["warmup_epochs"],
    cos_lr      = True,
    # Regularization
    patience    = HPARAMS["patience"],
    # Misc
    seed        = HPARAMS["seed"],
    amp         = HPARAMS["amp"],
    workers     = HPARAMS["num_workers"],
    device      = 0,                 # RTX 5060
    # Output
    project     = str(RUNS_DIR),
    name        = RUN_NAME,
    exist_ok    = True,
    save        = True,
    plots       = True,
    # Loss weights
    box         = HPARAMS["box"],
    cls         = HPARAMS["cls"],
    dfl         = HPARAMS["dfl"],
    # Augmentation — unpack AUG_CONFIG
    **AUG_CONFIG,
)

elapsed_v8m = time.time() - t0

# ─── Log & Summary ────────────────────────────────────────
log_experiment(
    model_name     = MODEL_NAME,
    hp             = HPARAMS,
    aug            = AUG_CONFIG,
    train_time_sec = elapsed_v8m,
    results_dir    = RUNS_DIR / RUN_NAME,
)
print_training_summary(MODEL_NAME, elapsed_v8m)

# ─── Lưu đường dẫn best.pt ────────────────────────────────
BEST_V8M = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
logger.info(f"📦 best.pt → {BEST_V8M}")
print(f"\n  best.pt : {BEST_V8M}")
print(f"  last.pt : {RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'}")


## tiếp train yolov8m

In [ ]:
# ============================================================
# CELL 12: TRAIN YOLOv8m — Target chính
# ============================================================
t0         = time.time()
MODEL_NAME = "yolov8m"
RUN_NAME   = f"{MODEL_NAME}_rdd2022"

model_v8m   = YOLO("yolov8m.pt")
logger.info(f"✅ Loaded: yolov8m.pt")

results_v8m = model_v8m.train(
    data          = str(DATA_YAML_PATH),
    epochs        = HPARAMS["num_epochs"],       # 50
    batch         = 8,                            # v8m nặng → batch nhỏ
    imgsz         = HPARAMS["img_size"],          # 800
    optimizer     = HPARAMS["optimizer"],         # AdamW
    lr0           = HPARAMS["learning_rate"],     # 0.001
    lrf           = HPARAMS["lr_final"] / HPARAMS["learning_rate"],  # 0.01
    weight_decay  = HPARAMS["weight_decay"],      # 0.0005
    momentum      = HPARAMS["momentum"],          # 0.937
    warmup_epochs = HPARAMS["warmup_epochs"],     # 3
    cos_lr        = True,
    patience      = HPARAMS["patience"],          # 15
    conf          = HPARAMS["conf_thres"],        # 0.001
    iou           = HPARAMS["iou_thres"],         # 0.6
    seed          = HPARAMS["seed"],              # 42
    amp           = HPARAMS["amp"],               # True
    workers       = HPARAMS["num_workers"],       # 4
    cache         = CACHE_MODE,
    box           = HPARAMS["box"],               # 7.5
    cls           = HPARAMS["cls"],               # 1.5
    dfl           = HPARAMS["dfl"],               # 1.5
    save_period   = HPARAMS["save_period"],       # 20
    device        = 0,
    project       = str(RUNS_DIR),
    name          = RUN_NAME,
    exist_ok      = True,
    save          = True,
    plots         = True,
    **AUG_CONFIG,
)

elapsed_v8m = time.time() - t0
BEST_V8M    = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
log_experiment(MODEL_NAME, HPARAMS, AUG_CONFIG, elapsed_v8m, RUNS_DIR / RUN_NAME)
print_training_summary(MODEL_NAME, elapsed_v8m)
logger.info(f"📦 best.pt → {BEST_V8M}")
print(f"  best.pt : {BEST_V8M}")
print(f"  last.pt : {RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'}")


## tiếp tục epoch 34

In [ ]:
# ============================================================
# CELL 12b: RESUME YOLOv8m — Tiếp tục từ epoch 34
# ============================================================
t0         = time.time()
MODEL_NAME = "yolov8m"
RUN_NAME   = f"{MODEL_NAME}_rdd2022"

# ─── Đường dẫn last.pt ────────────────────────────────────
LAST_V8M = RUNS_DIR / RUN_NAME / "weights" / "last.pt"

# ─── Kiểm tra file tồn tại trước khi resume ───────────────
if not LAST_V8M.exists():
    logger.error(f"❌ Không tìm thấy: {LAST_V8M}")
    logger.error("   → Kiểm tra lại đường dẫn RUNS_DIR")
else:
    logger.info(f"✅ Tìm thấy checkpoint: {LAST_V8M}")
    logger.info(f"🔄 Đang resume từ epoch đã dừng...")

    model_v8m   = YOLO(str(LAST_V8M))   # load last.pt

    results_v8m = model_v8m.train(
        resume = True,   # ← CHỈ CẦN DÒNG NÀY, YOLO tự đọc lại toàn bộ config cũ
    )

    elapsed_v8m = time.time() - t0
    BEST_V8M    = RUNS_DIR / RUN_NAME / "weights" / "best.pt"

    log_experiment(MODEL_NAME, HPARAMS, AUG_CONFIG, elapsed_v8m, RUNS_DIR / RUN_NAME)
    print_training_summary(MODEL_NAME, elapsed_v8m)
    logger.info(f"📦 best.pt → {BEST_V8M}")
    print(f"\n  best.pt : {BEST_V8M}")
    print(f"  last.pt : {RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'}")

In [ ]:
# ============================================================
# CELL 12b: RESUME YOLOv8m — Tiếp tục từ epoch 34
# ============================================================
t0         = time.time()
MODEL_NAME = "yolov8m"
RUN_NAME   = f"{MODEL_NAME}_rdd2022"

# ─── Đường dẫn last.pt ────────────────────────────────────
LAST_V8M = RUNS_DIR / RUN_NAME / "weights" / "last.pt"

# ─── Kiểm tra file tồn tại trước khi resume ───────────────
if not LAST_V8M.exists():
    logger.error(f"❌ Không tìm thấy: {LAST_V8M}")
    logger.error("   → Kiểm tra lại đường dẫn RUNS_DIR")
else:
    logger.info(f"✅ Tìm thấy checkpoint: {LAST_V8M}")
    logger.info(f"🔄 Đang resume từ epoch đã dừng...")

    model_v8m   = YOLO(str(LAST_V8M))   # load last.pt

    results_v8m = model_v8m.train(
        resume = True,   # ← CHỈ CẦN DÒNG NÀY, YOLO tự đọc lại toàn bộ config cũ
    )

    elapsed_v8m = time.time() - t0
    BEST_V8M    = RUNS_DIR / RUN_NAME / "weights" / "best.pt"

    log_experiment(MODEL_NAME, HPARAMS, AUG_CONFIG, elapsed_v8m, RUNS_DIR / RUN_NAME)
    print_training_summary(MODEL_NAME, elapsed_v8m)
    logger.info(f"📦 best.pt → {BEST_V8M}")
    print(f"\n  best.pt : {BEST_V8M}")
    print(f"  last.pt : {RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'}")

# yolo12m

In [ ]:
# ============================================================
# CELL XX: UPDATE ULTRALYTICS (chạy 1 lần duy nhất)
# ============================================================
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-U", "ultralytics"],
    capture_output=True, text=True
)
print(result.stdout[-2000:])   # in 2000 ký tự cuối

# Kiểm tra version sau update
import ultralytics
print(f"\n✅ ultralytics version: {ultralytics.__version__}")
# Cần >= 8.3.x để dùng yolo12


In [ ]:
# ============================================================
# CELL 12: TRAIN YOLO12m — Attention-based architecture
# ============================================================
t0         = time.time()
MODEL_NAME = "yolo12m"         # ← KHÔNG có chữ "v" (khác v8/v9/v10)
RUN_NAME   = f"{MODEL_NAME}_rdd2022"

model_v12m  = YOLO("yolo12m.pt")   # ← "yolo12m" không phải "yolov12m"
logger.info(f"✅ Loaded: yolo12m.pt")

results_v12m = model_v12m.train(
    data          = str(DATA_YAML_PATH),
    epochs        = HPARAMS["num_epochs"],
    batch         = 8,
    imgsz         = 640 ,
    optimizer     = HPARAMS["optimizer"],
    lr0           = HPARAMS["learning_rate"],
    lrf           = HPARAMS["lr_final"] / HPARAMS["learning_rate"],
    weight_decay  = HPARAMS["weight_decay"],
    momentum      = HPARAMS["momentum"],
    warmup_epochs = HPARAMS["warmup_epochs"],
    cos_lr        = True,
    patience      = HPARAMS["patience"],
    conf          = HPARAMS["conf_thres"],
    iou           = HPARAMS["iou_thres"],
    seed          = HPARAMS["seed"],
    amp           = HPARAMS["amp"],
    workers       = HPARAMS["num_workers"],
    cache         = CACHE_MODE,
    box           = HPARAMS["box"],
    cls           = HPARAMS["cls"],
    dfl           = HPARAMS["dfl"],
    save_period   = HPARAMS["save_period"],
    device        = 0,
    project       = str(RUNS_DIR),
    name          = RUN_NAME,
    exist_ok      = True,
    save          = True,
    plots         = True,
    **AUG_CONFIG,
)

elapsed_v12m = time.time() - t0
BEST_V12M    = RUNS_DIR / RUN_NAME / "weights" / "best.pt"
log_experiment(MODEL_NAME, HPARAMS, AUG_CONFIG, elapsed_v12m, RUNS_DIR / RUN_NAME)
print_training_summary(MODEL_NAME, elapsed_v12m)
logger.info(f"📦 best.pt → {BEST_V12M}")
print(f"  best.pt : {BEST_V12M}")
print(f"  last.pt : {RUNS_DIR / RUN_NAME / 'weights' / 'last.pt'}")


# TRAIN YOLOv8 - CONFIG MỚI

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🚀 TRAIN YOLOv8 - PATH CHÍNH XÁC (RDDD47k dataset)
# D:\Code\Data\RDDD47k_70_20_10\road_damage.yaml
# epochs=100, batch=16, lr0=0.01, erasing/mixup=0.0, SAVE MỖI 10 EPOCH
# ═══════════════════════════════════════════════════════════════
from ultralytics import YOLO
import time
import os

# Config với đường dẫn chính xác của bạn
DATA_PATH = r"D:\Code\Data\RDDD47k_70_20_10\road_damage.yaml"
MODELNAME = 'yolov8m'
RUNNAME = f'{MODELNAME}_fix_bg_d00_lr01_100e_b16'
PROJECT_DIR = r"D:\Code\Data\RDDD47k_70_20_10\runs\detect"

# ✅ Tạo output folder
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"📁 Data: {DATA_PATH}")
print(f"📁 Output: {PROJECT_DIR}/{RUNNAME}")

# Kiểm tra data.yaml tồn tại
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"❌ Không tìm thấy: {DATA_PATH}")
print("✅ Data.yaml OK")

# Load model (tự download yolov8m.pt)
model = YOLO('yolov8m.pt')
print("✅ Loaded yolov8m.pt")

# 🏃‍♂️ TRAIN - FULL CONFIG + SAVE MỖI 10 EPOCH ⭐
t0 = time.time()
results = model.train(
    data=DATA_PATH,                  # ✅ Đường dẫn của bạn

    # Training core ⭐
    epochs=100,                      # 100 epochs
    batch=16,                        # RTX 5060 safe
    imgsz=640,                       # Crack detail tốt

    # Optimizer ⭐
    lr0=0.01,                        # Tăng lr0 fix convergence
    lrf=0.01,                        # ✅ SỬA: 1e-5 thay vì 0.01 (quá cao!)
    momentum=0.937,
    weight_decay=0.0005,

    # Loss weights (imbalance D40/D00)
    box=7.5,
    cls=1.5,                         # Tăng cls weight
    dfl=1.5,

    # Augmentation ⭐ FIX BG/D00
    mosaic=1.0,                      # ⭐ Quan trọng nhất crack context
    mixup=0.0,                       # ❌ TẮT - tránh blend confusion
    close_mosaic=10,                 # Tắt mosaic cuối
    fliplr=0.5,                      # Crack 2 hướng
    flipud=0.0,                      # Không lật dọc
    hsv_h=0.015,
    hsv_s=0.7,                       # Multi-camera RDD
    hsv_v=0.4,
    degrees=5.0,
    shear=2.0,
    perspective=0.0005,
    scale=0.5,
    translate=0.1,
    copy_paste=0.1,                  # Minority D40
    erasing=0.0,                     # ❌ TẮT - tránh xóa boundary

    # Output & Validation ⭐ SAVE MỖI 10 EPOCH
    device=0,                        # RTX 5060
    workers=4,                       # Windows safe
    project=PROJECT_DIR,
    name=RUNNAME,
    exist_ok=True,
    save=True,
    save_period=10,                  # ⭐ SAVE MỖI 10 EPOCHS (10,20,30,...,100)
    plots=True,                      # Confusion Matrix tự động
    val=True,
    cache=False,
    amp=True,
    patience=15,
    seed=42,
    conf=0.001,
    iou=0.6                          # Crack mỏng
)

# Results
elapsed = time.time() - t0
BEST_PT = f"{PROJECT_DIR}/{RUNNAME}/weights/best.pt"
CONFUSION = f"{PROJECT_DIR}/{RUNNAME}/confusion_matrix.png"

print("\n" + "═" * 80)
print("✅ TRAINING HOÀN THÀNH!")
print(f"⏱️  Thời gian: {elapsed/60:.1f} phút")
print(f"⭐ BEST MODEL: {BEST_PT}")
print(f"📊 Confusion Matrix: {CONFUSION}")
print(f"📈 TensorBoard: tensorboard --logdir {PROJECT_DIR}")
print(f"💾 Checkpoints: epochs 10,20,30,...,100.pt")
print("═" * 80)

# Validate best model ngay
print("\n🔍 VALIDATE BEST MODEL...")
model = YOLO(BEST_PT)
metrics = model.val()
print(f"🎯 mAP50: {metrics.box.map50:.3f}  (↑0.65-0.70 expected)")
print(f"🎯 mAP50-95: {metrics.box.map:.3f}")

## tiếp tục từ 91epoch

In [ ]:
from ultralytics import YOLO
import time, os

DATA_PATH = r"D:\\Code\\Data\\RDDD47k_70_20_10\\road_damage.yaml"
MODELNAME = 'yolov8m'
RUNNAME = f'{MODELNAME}_fix_bg_d00_lr01_100e_b16'
PROJECT_DIR = r"D:\\Code\\Data\\RDDD47k_70_20_10\\runs\\detect"

LAST_PT = fr"{PROJECT_DIR}\\{RUNNAME}\\weights\\last.pt"

if os.path.exists(LAST_PT):
    print(f"🔄 Resume from: {LAST_PT}")
    model = YOLO(LAST_PT)
    resume_flag = True
else:
    print("🆕 Start from pretrained yolov8m.pt")
    model = YOLO('yolov8m.pt')
    resume_flag = False

t0 = time.time()
results = model.train(
    data=DATA_PATH,
    epochs=100,
    batch=16,
    imgsz=640,
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    box=7.5,
    cls=1.5,
    dfl=1.5,
    mosaic=1.0,
    mixup=0.0,
    close_mosaic=10,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    shear=2.0,
    perspective=0.0005,
    scale=0.5,
    translate=0.1,
    copy_paste=0.1,
    erasing=0.0,
    device=0,
    workers=4,
    project=PROJECT_DIR,
    name=RUNNAME,
    exist_ok=True,
    save=True,
    save_period=10,
    plots=True,
    val=True,
    cache=False,
    amp=True,
    patience=15,
    seed=42,
    conf=0.001,
    iou=0.6,
    resume=resume_flag,
)

In [ ]:
# Dùng conda
!conda install -c conda-forge ffmpeg

# Hoặc pip
!pip install ffmpeg-python

# test videos

In [ ]:
from ultralytics import YOLO
import cv2, os
from collections import defaultdict

# ===== CONFIG =====
MODEL_PATH      = r"D:\Code\Data\RDDD47k_70_20_10\runs\detect\yolov8m_fix_bg_d00_lr01_100e_b161\weights\best.pt"
VIDEO_INPUT_DIR = r"G:\My Drive\Môn Học trên lớp\Kì II năm 3\Học phần  Thị giác máy tính nâng cao\det\video_test"
OUTPUT_DIR      = r"G:\My Drive\Môn Học trên lớp\Kì II năm 3\Học phần  Thị giác máy tính nâng cao\det\video_test\output_video"

CONF = 0.25
IOU  = 0.5

os.makedirs(OUTPUT_DIR, exist_ok=True)

model = YOLO(MODEL_PATH)
all_class_names = list(model.names.values())

VIDEO_EXT = (".mp4", ".mov", ".avi", ".mkv")

video_files = [
    f for f in os.listdir(VIDEO_INPUT_DIR)
    if os.path.isfile(os.path.join(VIDEO_INPUT_DIR, f))
    and f.lower().endswith(VIDEO_EXT)
]

print(f"🎬 Found {len(video_files)} videos\n")

# ================= LOOP VIDEO =================
for file in sorted(video_files):
    in_path  = os.path.join(VIDEO_INPUT_DIR, file)
    out_path = os.path.join(OUTPUT_DIR, f"out_{os.path.splitext(file)[0]}.mp4")

    cap = cv2.VideoCapture(in_path)
    if not cap.isOpened():
        print(f"❌ Cannot open: {file}")
        continue

    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(
        out_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (w, h)
    )

    counted_ids = set()
    total_counts = defaultdict(int)

    print(f"▶️ Processing: {file} ({w}x{h} @ {fps:.0f}fps)")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # ===== TRACK =====
        results = model.track(
            frame,
            persist=True,
            conf=CONF,
            iou=IOU,
            tracker="bytetrack.yaml"
        )

        r = results[0]

        # nếu không có box → ghi luôn frame
        if r.boxes is None:
            writer.write(frame)
            continue

        keep_idx = []

        # ===== SAFE CHECK ID =====
        if r.boxes.id is not None:
            ids   = r.boxes.id.cpu().numpy().astype(int)
            clss  = r.boxes.cls.cpu().numpy().astype(int)
            confs = r.boxes.conf.cpu().numpy()

            for i, (obj_id, cls, conf) in enumerate(zip(ids, clss, confs)):
                name = model.names[cls]

                # 🔥 lọc D04 yếu
                if name == "D04" and conf < 0.5:
                    continue

                keep_idx.append(i)

                # ===== COUNT =====
                if obj_id not in counted_ids:
                    counted_ids.add(obj_id)
                    total_counts[name] += 1

        # ===== DRAW =====
        if len(keep_idx) > 0:
            boxes = r.boxes.xyxy.cpu().numpy()
            clss  = r.boxes.cls.cpu().numpy()
            confs = r.boxes.conf.cpu().numpy()

            for i in keep_idx:
                x1, y1, x2, y2 = map(int, boxes[i])
                name = model.names[int(clss[i])]
                conf = confs[i]

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
                cv2.putText(frame, f"{name} {conf:.2f}",
                            (x1, y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.7,
                            (0,255,0),
                            2)

        im = frame

        # ===== UI =====
        total_all = sum(total_counts.values())

        overlay = im.copy()
        cv2.rectangle(overlay, (10, 10), (450, 300), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.6, im, 0.4, 0, im)

        cv2.putText(im, f"TOTAL: {total_all}",
                    (20, 60),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.8,
                    (0, 0, 255),
                    4,
                    cv2.LINE_AA)

        y = 110
        for name in sorted(all_class_names):
            cnt = total_counts[name]
            color = (0, 255, 120) if cnt > 0 else (180, 180, 180)

            cv2.putText(im, f"{name}: {cnt}",
                        (20, y),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        1.1,
                        color,
                        3,
                        cv2.LINE_AA)
            y += 40

        writer.write(im)

    cap.release()
    writer.release()

    print(f"✅ Done: {file}")
    print(f"📊 Total: {dict(total_counts)}\n")

print("🏁 DONE ALL VIDEOS")

In [ ]:
import imageio_ffmpeg

# Lấy đường dẫn ffmpeg.exe được cài kèm imageio_ffmpeg
FFMPEG_EXE = imageio_ffmpeg.get_ffmpeg_exe()
print(f"✅ ffmpeg tìm thấy tại: {FFMPEG_EXE}")

In [ ]:
ret = subprocess.run(
    [
        FFMPEG_EXE,          # ⭐ thay "ffmpeg" bằng FFMPEG_EXE
        "-i", in_path,
        "-c:v", "libx264",
        "-crf", "18",
        "-preset", "fast",
        "-c:a", "aac",
        "-y",
        out_path
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# test images

In [ ]:
from ultralytics import YOLO
import cv2, os
from collections import defaultdict

# ===== CONFIG =====
MODEL_PATH      = r"D:\Code\Data\RDDD47k_70_20_10\runs\detect\yolov8m_fix_bg_d00_lr01_100e_b161\weights\best.pt"
IMAGE_INPUT_DIR = r"G:\My Drive\Môn Học trên lớp\Kì II năm 3\Học phần  Thị giác máy tính nâng cao\det\test_image"
OUTPUT_DIR      = r"G:\My Drive\Môn Học trên lớp\Kì II năm 3\Học phần  Thị giác máy tính nâng cao\det\test_image\output_image"

CONF = 0.25
IOU  = 0.5

os.makedirs(OUTPUT_DIR, exist_ok=True)

model = YOLO(MODEL_PATH)
all_class_names = list(model.names.values())

IMAGE_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

image_files = [
    f for f in os.listdir(IMAGE_INPUT_DIR)
    if os.path.isfile(os.path.join(IMAGE_INPUT_DIR, f))
    and f.lower().endswith(IMAGE_EXT)
]

print(f"🖼️ Found {len(image_files)} images\n")

# ===== TOTAL COUNT ALL IMAGES =====
total_counts_all_images = defaultdict(int)

# ================= LOOP IMAGE =================
for file in sorted(image_files):
    in_path  = os.path.join(IMAGE_INPUT_DIR, file)
    out_path = os.path.join(OUTPUT_DIR, f"out_{file}")

    img = cv2.imread(in_path)
    if img is None:
        print(f"❌ Cannot read: {file}")
        continue

    h, w = img.shape[:2]
    print(f"▶️ Processing: {file} ({w}x{h})")

    # ===== PREDICT =====
    results = model.predict(
        source=img,
        conf=CONF,
        iou=IOU,
        verbose=False
    )

    r = results[0]

    image_counts = defaultdict(int)

    # nếu không có box → vẫn lưu ảnh gốc
    if r.boxes is not None and len(r.boxes) > 0:
        boxes = r.boxes.xyxy.cpu().numpy()
        clss  = r.boxes.cls.cpu().numpy().astype(int)
        confs = r.boxes.conf.cpu().numpy()

        for i, (cls, conf) in enumerate(zip(clss, confs)):
            name = model.names[cls]

            # 🔥 lọc D04 yếu
            if name == "D04" and conf < 0.5:
                continue

            x1, y1, x2, y2 = map(int, boxes[i])

            # ===== COUNT =====
            image_counts[name] += 1
            total_counts_all_images[name] += 1

            # ===== DRAW =====
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                img,
                f"{name} {conf:.2f}",
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

    # ===== UI =====
    total_all = sum(image_counts.values())

    overlay = img.copy()
    cv2.rectangle(overlay, (10, 10), (450, 300), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.6, img, 0.4, 0, img)

    cv2.putText(
        img,
        f"TOTAL: {total_all}",
        (20, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.5,
        (0, 0, 255),
        3,
        cv2.LINE_AA
    )

    y = 110
    for name in sorted(all_class_names):
        cnt = image_counts[name]
        color = (0, 255, 120) if cnt > 0 else (180, 180, 180)

        cv2.putText(
            img,
            f"{name}: {cnt}",
            (20, y),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            color,
            2,
            cv2.LINE_AA
        )
        y += 35

    cv2.imwrite(out_path, img)

    print(f"✅ Done: {file}")
    print(f"📊 Count in image: {dict(image_counts)}\n")

print("🏁 DONE ALL IMAGES")
print(f"📦 TOTAL ALL IMAGES: {dict(total_counts_all_images)}")

#  IMPORT & DEPENDENCIES

In [ ]:
# ============================================================
# CELL BG-01: IMPORT & DEPENDENCIES
# ============================================================
import os
import cv2
import numpy as np
import shutil
import random
from pathlib import Path
from tqdm import tqdm
import albumentations as A
from PIL import Image, ImageEnhance, ImageFilter
import warnings
warnings.filterwarnings("ignore")

# Kiểm tra albumentations
try:
    import albumentations as A
    print(f"✅ albumentations version: {A.__version__}")
except ImportError:
    print("❌ Chưa cài albumentations. Chạy: pip install albumentations")

print("✅ Import xong tất cả dependencies")

In [ ]:
# ============================================================
# CELL BG-02: PATHS CONFIG (khớp với dataset gốc của bạn)
# ============================================================
from pathlib import Path

# ─── Dataset gốc ─────────────────────────────────────────
BDD_ROOT      = Path(r"D:\Code\Data\RDDD47k_70_20_10")
TRAIN_IMG_DIR = BDD_ROOT / "train" / "images"
TRAIN_LBL_DIR = BDD_ROOT / "train" / "labels"

# ─── Thư mục chứa ảnh nền gốc bạn thu thập (chưa augment) ─
# Để đây những ảnh mặt đường ban đêm RAW bạn chụp/tải về
BG_RAW_DIR    = BDD_ROOT / "bg_raw_night"   # ảnh nền thô bạn cung cấp

# ─── Thư mục output ảnh nền đã augment (trước khi copy vào train) ─
BG_AUG_DIR    = BDD_ROOT / "bg_augmented"   # kết quả sau augmentation

# Tạo thư mục nếu chưa có
BG_RAW_DIR.mkdir(parents=True, exist_ok=True)
BG_AUG_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 BDD_ROOT      : {BDD_ROOT}")
print(f"📁 TRAIN_IMG_DIR : {TRAIN_IMG_DIR}")
print(f"📁 TRAIN_LBL_DIR : {TRAIN_LBL_DIR}")
print(f"📁 BG_RAW_DIR    : {BG_RAW_DIR}  <-- Đặt ảnh nền gốc vào đây")
print(f"📁 BG_AUG_DIR    : {BG_AUG_DIR}  <-- Ảnh augmented sẽ xuất ra đây")

# ─── Cấu hình augmentation ────────────────────────────────
BG_CONFIG = {
    "n_augment_per_image" : 6,     # Mỗi ảnh gốc tạo ra 6 biến thể
    "target_total"        : 500,   # Tổng số ảnh nền muốn có (tự động dừng)
    "img_size"            : 640,   # Resize về 640x640 (khớp imgsz YOLO)
    "seed"                : 42,
}

random.seed(BG_CONFIG["seed"])
np.random.seed(BG_CONFIG["seed"])
print(f"\n⚙️  Mỗi ảnh gốc → {BG_CONFIG['n_augment_per_image']} biến thể")
print(f"⚙️  Tổng mục tiêu : {BG_CONFIG['target_total']} ảnh background"
      )

## ĐỊNH NGHĨA CÁC PIPELINE AUGMENTATION

In [ ]:
# ============================================================
# CELL BG-03: ĐỊNH NGHĨA CÁC PIPELINE AUGMENTATION
#             (Giả lập điều kiện ban đêm đa dạng)
# ============================================================

# ── Pipeline 1: Đêm tối thông thường ──────────────────────
pipeline_dark_night = A.Compose([
    A.RandomBrightnessContrast(
        brightness_limit=(-0.5, -0.2),   # Tối hơn đáng kể
        contrast_limit=(-0.1, 0.3),
        p=1.0
    ),
    A.HueSaturationValue(
        hue_shift_limit=5,
        sat_shift_limit=(-40, -10),       # Giảm màu sắc (đêm ít màu)
        val_shift_limit=(-50, -20),       # Giảm độ sáng tổng thể
        p=0.9
    ),
    A.GaussNoise(var_limit=(20, 80), p=0.6),   # Nhiễu cảm biến camera đêm
    A.Blur(blur_limit=(3, 5), p=0.3),
])

# ── Pipeline 2: Đèn đường vàng / sodium lamp ──────────────
pipeline_streetlamp = A.Compose([
    A.RandomBrightnessContrast(
        brightness_limit=(-0.3, 0.1),
        contrast_limit=(0.1, 0.4),
        p=1.0
    ),
    A.HueSaturationValue(
        hue_shift_limit=15,               # Shift hue sang vàng cam
        sat_shift_limit=(-20, 20),
        val_shift_limit=(-20, 30),
        p=0.9
    ),
    A.RandomShadow(
        shadow_roi=(0, 0.3, 1, 1),        # Bóng từ giữa ảnh trở xuống
        num_shadows_lower=1,
        num_shadows_upper=4,
        shadow_dimension=6,
        p=0.85
    ),
    A.GaussNoise(var_limit=(10, 50), p=0.5),
])

# ── Pipeline 3: Bóng cây / bóng xe đậm đặc ───────────────
pipeline_heavy_shadow = A.Compose([
    A.RandomShadow(
        shadow_roi=(0, 0, 1, 1),          # Bóng toàn ảnh
        num_shadows_lower=2,
        num_shadows_upper=6,              # Nhiều vùng bóng chồng chéo
        shadow_dimension=8,
        p=1.0
    ),
    A.RandomBrightnessContrast(
        brightness_limit=(-0.35, 0.0),
        contrast_limit=(0.0, 0.35),
        p=0.8
    ),
    A.HueSaturationValue(
        hue_shift_limit=8,
        sat_shift_limit=(-25, 5),
        val_shift_limit=(-35, 10),
        p=0.7
    ),
])

# ── Pipeline 4: Đèn pha xe chiếu ngược (glare) ────────────
pipeline_headlight_glare = A.Compose([
    A.RandomBrightnessContrast(
        brightness_limit=(-0.2, 0.4),    # Một số vùng rất sáng
        contrast_limit=(0.2, 0.5),
        p=1.0
    ),
    A.RandomShadow(
        shadow_roi=(0.2, 0.5, 0.8, 1),   # Bóng tối phần dưới ảnh
        num_shadows_lower=1,
        num_shadows_upper=3,
        shadow_dimension=5,
        p=0.8
    ),
    A.GaussianBlur(blur_limit=(3, 7), p=0.4),   # Halo effect
    A.HueSaturationValue(
        hue_shift_limit=20,
        sat_shift_limit=(-10, 30),
        val_shift_limit=(-10, 40),
        p=0.6
    ),
])

# ── Pipeline 5: Mưa / đường ướt ban đêm ──────────────────
pipeline_wet_rainy = A.Compose([
    A.RandomRain(
        slant_lower=-10, slant_upper=10,
        drop_length=15,  drop_width=1,
        drop_color=(180, 180, 180),
        blur_value=2,
        brightness_coefficient=0.7,
        rain_type="drizzle",
        p=0.8
    ),
    A.RandomBrightnessContrast(
        brightness_limit=(-0.4, -0.1),
        contrast_limit=(-0.1, 0.2),
        p=0.9
    ),
    A.HueSaturationValue(
        hue_shift_limit=5,
        sat_shift_limit=(-30, 0),         # Màu xám khi mưa
        val_shift_limit=(-40, -10),
        p=0.8
    ),
    A.GaussNoise(var_limit=(15, 60), p=0.6),
])

# ── Pipeline 6: Sương mù đêm ─────────────────────────────
pipeline_foggy_night = A.Compose([
    A.RandomFog(
        fog_coef_lower=0.2,
        fog_coef_upper=0.6,
        alpha_coef=0.1,
        p=0.9
    ),
    A.RandomBrightnessContrast(
        brightness_limit=(-0.3, 0.0),
        contrast_limit=(-0.3, 0.0),       # Giảm contrast trong sương
        p=0.9
    ),
    A.GaussianBlur(blur_limit=(3, 9), p=0.6),
])

# Danh sách tất cả pipelines kèm tên
ALL_PIPELINES = [
    ("dark_night",       pipeline_dark_night),
    ("streetlamp",       pipeline_streetlamp),
    ("heavy_shadow",     pipeline_heavy_shadow),
    ("headlight_glare",  pipeline_headlight_glare),
    ("wet_rainy",        pipeline_wet_rainy),
    ("foggy_night",      pipeline_foggy_night),
]

print(f"✅ Đã định nghĩa {len(ALL_PIPELINES)} pipeline augmentation:")
for name, _ in ALL_PIPELINES:
    print(f"   • {name}")

In [ ]:
# ============================================================
# CELL BG-04: HÀM HELPER
# ============================================================

def load_and_resize(img_path: Path, size: int = 640) -> np.ndarray:
    """Load ảnh và resize về size x size."""
    img = cv2.imread(str(img_path))
    if img is None:
        raise ValueError(f"Không đọc được ảnh: {img_path}")
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_LINEAR)
    return img  # BGR

def apply_pipeline(img_bgr: np.ndarray, pipeline) -> np.ndarray:
    """Apply một Albumentations pipeline lên ảnh BGR."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    result  = pipeline(image=img_rgb)["image"]
    return cv2.cvtColor(result, cv2.COLOR_RGB2BGR)

def count_existing_bg(train_lbl_dir: Path) -> int:
    """Đếm số file .txt rỗng (background) trong thư mục labels."""
    count = 0
    for f in train_lbl_dir.glob("*.txt"):
        if f.stat().st_size == 0:
            count += 1
    return count

def verify_bg_consistency(train_img_dir: Path, train_lbl_dir: Path):
    """Kiểm tra mỗi ảnh bg có file .txt rỗng tương ứng không."""
    issues = []
    for img_f in train_img_dir.glob("bg_*"):
        lbl_f = train_lbl_dir / (img_f.stem + ".txt")
        if not lbl_f.exists():
            issues.append(f"THIẾU label: {lbl_f.name}")
        elif lbl_f.stat().st_size > 0:
            issues.append(f"Label KHÔNG rỗng: {lbl_f.name}")
    return issues

print("✅ Các hàm helper đã sẵn sàng")

## TỰ SINH ẢNH ĐÊM TỪ ẢNH NGÀY CÓ SẴN

In [ ]:
# ============================================================
# CELL BG-04b: TỰ SINH ẢNH ĐÊM TỪ ẢNH NGÀY CÓ SẴN
# Lấy ảnh background ban ngày trong train/ → chuyển thành đêm
# → Lưu vào BG_RAW_DIR để BG-05 dùng tiếp
# ============================================================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Pipeline "Day → Night" mạnh tay
day_to_night = A.Compose([
    # Tối hoá ảnh rất mạnh
    A.RandomBrightnessContrast(
        brightness_limit=(-0.65, -0.40),  # Giảm sáng 40-65%
        contrast_limit=(-0.2, 0.3),
        p=1.0
    ),
    # Giảm saturation → màu đêm xám/lạnh
    A.HueSaturationValue(
        hue_shift_limit=10,
        sat_shift_limit=(-60, -20),       # Mất màu đặc trưng ban đêm
        val_shift_limit=(-60, -30),
        p=1.0
    ),
    # Thêm nhiễu ISO camera đêm
    A.GaussNoise(var_limit=(30, 100), p=0.8),
    # Nhòe nhẹ (camera đêm thường blur)
    A.GaussianBlur(blur_limit=(3, 5), p=0.5),
    # Tone màu lạnh (ánh đèn LED/đường phố)
    A.RGBShift(
        r_shift_limit=(-10, 5),
        g_shift_limit=(-10, 5),
        b_shift_limit=(0, 20),            # Tăng blue → tone đêm lạnh
        p=0.6
    ),
])

# ── Lấy ảnh nguồn: ưu tiên ảnh background ban ngày ────────
# (file .txt rỗng = background, không có ổ gà → an toàn để dùng làm nền đêm)
source_images = []

for lbl_f in TRAIN_LBL_DIR.glob("*.txt"):
    if lbl_f.stat().st_size == 0:           # Chỉ lấy background
        for ext in IMG_EXTS:
            img_f = TRAIN_IMG_DIR / (lbl_f.stem + ext)
            if img_f.exists():
                source_images.append(img_f)
                break

# Nếu không đủ background, lấy thêm ảnh có nhãn (nhưng chỉ lấy nền)
if len(source_images) < 50:
    all_imgs = list(TRAIN_IMG_DIR.glob("*.jpg")) + list(TRAIN_IMG_DIR.glob("*.png"))
    random.shuffle(all_imgs)
    source_images += all_imgs[:200 - len(source_images)]

random.shuffle(source_images)
print(f"📂 Nguồn ảnh ban ngày: {len(source_images)} ảnh")

# ── Giới hạn số ảnh nguồn cần xử lý ──────────────────────
N_SOURCE = min(100, len(source_images))   # Lấy tối đa 100 ảnh nguồn
source_images = source_images[:N_SOURCE]

# ── Chuyển đổi Day → Night và lưu vào BG_RAW_DIR ──────────
converted = 0
errors    = 0

for src_img_path in tqdm(source_images, desc="Day → Night conversion"):
    try:
        img = cv2.imread(str(src_img_path))
        if img is None:
            errors += 1
            continue

        img_resized = cv2.resize(img, (640, 640))
        img_rgb     = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)

        night_rgb = day_to_night(image=img_rgb)["image"]
        night_bgr = cv2.cvtColor(night_rgb, cv2.COLOR_RGB2BGR)

        # Lưu vào BG_RAW_DIR với prefix "synth_night_"
        out_name = f"synth_night_{src_img_path.stem}.jpg"
        out_path = BG_RAW_DIR / out_name
        cv2.imwrite(str(out_path), night_bgr, [cv2.IMWRITE_JPEG_QUALITY, 92])
        converted += 1

    except Exception as e:
        print(f"⚠️  Lỗi {src_img_path.name}: {e}")
        errors += 1

print(f"\n✅ Đã tạo {converted} ảnh ban đêm giả lập → lưu vào BG_RAW_DIR")
print(f"❌ Lỗi: {errors}")
print(f"\n→ Bây giờ chạy tiếp CELL BG-05 để tạo 500 biến thể đêm đa dạng")

In [ ]:
# ============================================================
# CELL BG-05: SINH ẢNH BACKGROUND VỚI AUGMENTATION
# ============================================================
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

raw_images = sorted([
    p for p in BG_RAW_DIR.rglob("*")
    if p.suffix.lower() in IMG_EXTS
])

if len(raw_images) == 0:
    print("⚠️  BG_RAW_DIR trống!")
    print(f"   → Hãy đặt ảnh nền ban đêm vào: {BG_RAW_DIR}")
    print("   → Ảnh nên là: mặt đường ban đêm, bóng cây/xe, KHÔNG có ổ gà")
else:
    print(f"📂 Tìm thấy {len(raw_images)} ảnh gốc trong BG_RAW_DIR")
    print(f"📊 Sẽ tạo tối đa {BG_CONFIG['target_total']} ảnh augmented\n")

    generated    = 0
    errors       = 0
    target_total = BG_CONFIG["target_total"]
    size         = BG_CONFIG["img_size"]

    pbar = tqdm(total=target_total, desc="Generating BG images", unit="img")

    for raw_img_path in raw_images:
        if generated >= target_total:
            break
        try:
            img_bgr = load_and_resize(raw_img_path, size)
        except Exception as e:
            print(f"⚠️  Lỗi load {raw_img_path.name}: {e}")
            errors += 1
            continue

        # Shuffle pipeline để không bị trùng thứ tự
        shuffled_pipelines = ALL_PIPELINES.copy()
        random.shuffle(shuffled_pipelines)

        for i, (pipe_name, pipeline) in enumerate(shuffled_pipelines):
            if generated >= target_total:
                break
            if i >= BG_CONFIG["n_augment_per_image"]:
                break
            try:
                aug_img = apply_pipeline(img_bgr, pipeline)
            except Exception as e:
                print(f"⚠️  Pipeline {pipe_name} lỗi: {e}")
                errors += 1
                continue

            # Đặt tên: bg_{tên gốc}_{pipeline}_{index}.jpg
            out_name = f"bg_{raw_img_path.stem}_{pipe_name}_{i:03d}.jpg"
            out_path = BG_AUG_DIR / out_name
            cv2.imwrite(str(out_path), aug_img, [cv2.IMWRITE_JPEG_QUALITY, 92])
            generated += 1
            pbar.update(1)

    pbar.close()
    print(f"\n✅ Tạo thành công : {generated} ảnh background")
    print(f"❌ Lỗi           : {errors}")
    print(f"📁 Lưu tại       : {BG_AUG_DIR}")

## COPY VÀO TRAIN + TẠO FILE .TXT RỖNG

In [ ]:
# ============================================================
# CELL BG-06: COPY VÀO TRAIN + TẠO FILE .TXT RỖNG
# ============================================================
aug_images = sorted(BG_AUG_DIR.glob("*.jpg"))

if len(aug_images) == 0:
    print("⚠️  BG_AUG_DIR trống — hãy chạy CELL BG-05 trước")
else:
    print(f"📂 {len(aug_images)} ảnh augmented sẵn sàng copy vào train")
    copied  = 0
    skipped = 0

    for aug_img_path in tqdm(aug_images, desc="Copying to train/"):
        dest_img = TRAIN_IMG_DIR / aug_img_path.name
        if dest_img.exists():
            dest_img = TRAIN_IMG_DIR / (aug_img_path.stem + "_dup.jpg")
            if dest_img.exists():
                skipped += 1
                continue

        shutil.copy2(aug_img_path, dest_img)

        # File .txt RỖNG = YOLO hiểu là "background image"
        # Model bị phạt nếu detect false positive → học rằng bóng đổ ≠ ổ gà
        label_path = TRAIN_LBL_DIR / (dest_img.stem + ".txt")
        label_path.touch()   # 0 bytes

        copied += 1

    print(f"\n✅ Copy thành công : {copied} ảnh + {copied} file .txt rỗng")
    print(f"⚠️  Bỏ qua (đã có) : {skipped}")

##  VERIFY & THỐNG KÊ CUỐI CÙNG

In [ ]:
# ============================================================
# CELL BG-07: VERIFY & THỐNG KÊ CUỐI CÙNG
# ============================================================
total_train_imgs = len(list(TRAIN_IMG_DIR.glob("*")))
total_bg_files   = count_existing_bg(TRAIN_LBL_DIR)
total_labeled    = total_train_imgs - total_bg_files

print("=" * 55)
print("📊 THỐNG KÊ DATASET SAU KHI THÊM BACKGROUND IMAGES")
print("=" * 55)
print(f"  Tổng ảnh train     : {total_train_imgs:,}")
print(f"  Ảnh có nhãn (ổ gà) : {total_labeled:,}  ({total_labeled/total_train_imgs*100:.1f}%)")
print(f"  Ảnh background     : {total_bg_files:,}  ({total_bg_files/total_train_imgs*100:.1f}%)")
print("=" * 55)

issues = verify_bg_consistency(TRAIN_IMG_DIR, TRAIN_LBL_DIR)
if issues:
    print(f"\n⚠️  Phát hiện {len(issues)} vấn đề:")
    for issue in issues[:10]:
        print(f"   • {issue}")
else:
    print("\n✅ Tất cả ảnh background đều có file .txt rỗng tương ứng")

# Gợi ý tỉ lệ lý tưởng
current_pct = total_bg_files / total_train_imgs * 100
if current_pct < 10:
    print(f"\n💡 Tỉ lệ BG ({current_pct:.1f}%) hơi thấp. Nên có 10-20%.")
elif current_pct > 25:
    print(f"\n💡 Tỉ lệ BG ({current_pct:.1f}%) hơi cao. Quá nhiều có thể giảm recall.")
else:
    print(f"\n✅ Tỉ lệ background {current_pct:.1f}% — nằm trong vùng lý tưởng (10-25%)")

### XÓA CACHE & SẴN SÀNG TRAIN

In [ ]:
# ============================================================
# CELL BG-08: XÓA CACHE & SẴN SÀNG TRAIN
# ============================================================
cache_files = list(BDD_ROOT.glob("*.cache")) + list(BDD_ROOT.rglob("*.cache"))

if cache_files:
    for cf in cache_files:
        cf.unlink()
        print(f"🗑️  Đã xóa cache: {cf}")
else:
    print("ℹ️  Không tìm thấy file .cache nào — bỏ qua")

print()
print("=" * 55)
print("✅ HOÀN TẤT — SẴN SÀNG TRAIN")
print("=" * 55)
print()
print("BƯỚC TIẾP THEO — Sửa CELL 09 Train:")
print("  • RUNNAME = 'yolov8m_bgd_shadow_v2'")
print("  • epochs  = 50  (fine-tune thêm từ best.pt)")
print("  • lr0     = 0.0005  (nhỏ hơn khi fine-tune)")
print("  • model   = YOLO('...best.pt')  ← load checkpoint cũ")

 #  YOLOv8

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 🚀 TRAIN YOLOv8 - PATH CHÍNH XÁC (RDDD47k dataset)
# D:\Code\Data\RDDD47k_70_20_10\road_damage.yaml
# epochs=100, batch=16, lr0=0.01, erasing/mixup=0.0, SAVE MỖI 10 EPOCH
# ═══════════════════════════════════════════════════════════════
from ultralytics import YOLO
import time
import os

# Config với đường dẫn chính xác của bạn
DATA_PATH = r"D:\Code\Data\RDDD47k_70_20_10\road_damage.yaml"
MODELNAME = 'yolov8m'
RUNNAME = f'{MODELNAME}_fix_bg_d00_lr01_100e_b161'
PROJECT_DIR = r"D:\Code\Data\RDDD47k_70_20_10\runs\detect"

# ✅ Tạo output folder
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"📁 Data: {DATA_PATH}")
print(f"📁 Output: {PROJECT_DIR}/{RUNNAME}")

# Kiểm tra data.yaml tồn tại
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"❌ Không tìm thấy: {DATA_PATH}")
print("✅ Data.yaml OK")

# Load model (tự download yolov8m.pt)
model = YOLO('yolov8m.pt')
print("✅ Loaded yolov8m.pt")

# 🏃‍♂️ TRAIN - FULL CONFIG + SAVE MỖI 10 EPOCH ⭐
t0 = time.time()
results = model.train(
    data=DATA_PATH,                  # ✅ Đường dẫn của bạn

    # Training core ⭐
    epochs=100,                      # 100 epochs
    batch=16,                        # RTX 5060 safe
    imgsz=640,                       # Crack detail tốt

    # Optimizer ⭐
    lr0=0.01,                        # Tăng lr0 fix convergence
    lrf=0.01,                        # ✅ SỬA: 1e-5 thay vì 0.01 (quá cao!)
    momentum=0.937,
    weight_decay=0.0005,

    # Loss weights (imbalance D40/D00)
    box=7.5,
    cls=1.5,                         # Tăng cls weight
    dfl=1.5,

    # Augmentation ⭐ FIX BG/D00
    mosaic=1.0,                      # ⭐ Quan trọng nhất crack context
    mixup=0.0,                       # ❌ TẮT - tránh blend confusion
    close_mosaic=10,                 # Tắt mosaic cuối
    fliplr=0.5,                      # Crack 2 hướng
    flipud=0.0,                      # Không lật dọc
    hsv_h=0.015,
    hsv_s=0.7,                       # Multi-camera RDD
    hsv_v=0.4,
    degrees=5.0,
    shear=2.0,
    perspective=0.0005,
    scale=0.5,
    translate=0.1,
    copy_paste=0.1,                  # Minority D40
    erasing=0.0,                     # ❌ TẮT - tránh xóa boundary

    # Output & Validation ⭐ SAVE MỖI 10 EPOCH
    device=0,                        # RTX 5060
    workers=4,                       # Windows safe
    project=PROJECT_DIR,
    name=RUNNAME,
    exist_ok=True,
    save=True,
    save_period=10,                  # ⭐ SAVE MỖI 10 EPOCHS (10,20,30,...,100)
    plots=True,                      # Confusion Matrix tự động
    val=True,
    cache=False,
    amp=True,
    patience=15,
    seed=42,
    conf=0.001,
    iou=0.6                          # Crack mỏng
)

# Results
elapsed = time.time() - t0
BEST_PT = f"{PROJECT_DIR}/{RUNNAME}/weights/best.pt"
CONFUSION = f"{PROJECT_DIR}/{RUNNAME}/confusion_matrix.png"

print("\n" + "═" * 80)
print("✅ TRAINING HOÀN THÀNH!")
print(f"⏱️  Thời gian: {elapsed/60:.1f} phút")
print(f"⭐ BEST MODEL: {BEST_PT}")
print(f"📊 Confusion Matrix: {CONFUSION}")
print(f"📈 TensorBoard: tensorboard --logdir {PROJECT_DIR}")
print(f"💾 Checkpoints: epochs 10,20,30,...,100.pt")
print("═" * 80)

# Validate best model ngay
print("\n🔍 VALIDATE BEST MODEL...")
model = YOLO(BEST_PT)
metrics = model.val()
print(f"🎯 mAP50: {metrics.box.map50:.3f}  (↑0.65-0.70 expected)")
print(f"🎯 mAP50-95: {metrics.box.map:.3f}")

## tiếp tục 91 epoch


In [ ]:
from ultralytics import YOLO
import time, os

DATA_PATH = r"D:\\Code\\Data\\RDDD47k_70_20_10\\road_damage.yaml"
MODELNAME = 'yolov8m'
RUNNAME = f'{MODELNAME}_fix_bg_d00_lr01_100e_b161'
PROJECT_DIR = r"D:\\Code\\Data\\RDDD47k_70_20_10\\runs\\detect"

LAST_PT = fr"{PROJECT_DIR}\\{RUNNAME}\\weights\\last.pt"

if os.path.exists(LAST_PT):
    print(f"🔄 Resume from: {LAST_PT}")
    model = YOLO(LAST_PT)
    resume_flag = True
else:
    print("🆕 Start from pretrained yolov8m.pt")
    model = YOLO('yolov8m.pt')
    resume_flag = False

t0 = time.time()
results = model.train(
    data=DATA_PATH,
    epochs=100,
    batch=16,
    imgsz=640,
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    box=7.5,
    cls=1.5,
    dfl=1.5,
    mosaic=1.0,
    mixup=0.0,
    close_mosaic=10,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    shear=2.0,
    perspective=0.0005,
    scale=0.5,
    translate=0.1,
    copy_paste=0.1,
    erasing=0.0,
    device=0,
    workers=4,
    project=PROJECT_DIR,
    name=RUNNAME,
    exist_ok=True,
    save=True,
    save_period=10,
    plots=True,
    val=True,
    cache=False,
    amp=True,
    patience=15,
    seed=42,
    conf=0.001,
    iou=0.6,
    resume=resume_flag,
)